In [ ]:
from google.colab import files
print("Select your train-balanced-sarcasm.csv file...")
uploaded = files.upload()

Select your train-balanced-sarcasm.csv file...


Saving final_dataset.csv to final_dataset.csv


In [ ]:
!pip install -q torch transformers scikit-learn pandas numpy spacy tqdm sentencepiece stanza
!python -m spacy download en_core_web_sm -q

import stanza
stanza.download('en', verbose=False)

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print("✅ GPU is ready! Training will be FAST!")
else:
    print("⚠️ No GPU found. Training will be slow.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 25.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 121.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
✅ GPU is ready! Training will be FAST!


In [ ]:
# ══════════════════════════════════════════════════════════════
#  HYBRID SARCASM PIPELINE v2 — FOR DATASET WITH:
#  COLUMNS = text, label
# ══════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import os
import re
import pickle
import spacy
import stanza
from collections import Counter
from transformers import (
    AutoModel, AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.utils.data import DataLoader, Dataset
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, accuracy_score, f1_score,
    confusion_matrix
)
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ══════════════════════════════════════════════════════════════
#  SETTINGS
# ══════════════════════════════════════════════════════════════

MAX_SAMPLES = 29970   # your dataset is already balanced ~14985 + 14985
NUM_EPOCHS = 5
BATCH_SIZE = 32
MAX_LENGTH = 128
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
MODEL_NAME = "roberta-base"
GRADIENT_ACCUM = 2
SAVE_PATH = "./saved_models/hybrid_v2"
os.makedirs(SAVE_PATH, exist_ok=True)

print(f"\nSettings:")
print(f"  Model:      {MODEL_NAME}")
print(f"  Samples:    {MAX_SAMPLES}")
print(f"  Epochs:     {NUM_EPOCHS}")
print(f"  Batch:      {BATCH_SIZE} (effective: {BATCH_SIZE * GRADIENT_ACCUM})")
print(f"  Max Length: {MAX_LENGTH}")
print(f"  LR:         {LEARNING_RATE}")

# ══════════════════════════════════════════════════════════════
#  ENHANCED LINGUISTIC FEATURES
# ══════════════════════════════════════════════════════════════

class LinguisticFeatureExtractor:
    def __init__(self):
        self.sarcasm_markers = {
            "oh", "wow", "sure", "right", "yeah", "totally",
            "obviously", "clearly", "definitely", "absolutely",
            "brilliant", "genius", "wonderful", "fantastic",
            "great", "amazing", "shocking", "surprise",
            "thanks", "thank", "love", "lovely", "nice",
            "perfect", "exactly", "precisely", "bravo",
            "yay", "hooray", "gee", "golly", "neat"
        }
        self.intensifiers = {
            "very", "really", "so", "extremely", "incredibly",
            "absolutely", "totally", "completely", "utterly",
            "truly", "deeply", "highly", "super", "literally",
            "just", "especially", "particularly", "remarkably"
        }
        self.negations = {
            "not", "no", "never", "neither", "nobody", "nothing",
            "nowhere", "nor", "none", "don't", "doesn't", "didn't",
            "won't", "wouldn't", "couldn't", "shouldn't", "can't",
            "isn't", "aren't", "wasn't", "weren't", "haven't",
            "hasn't", "hadn't"
        }
        self.positive_words = {
            "love", "great", "good", "best", "happy", "wonderful",
            "excellent", "amazing", "awesome", "fantastic", "perfect",
            "beautiful", "brilliant", "nice", "enjoy", "enjoyed",
            "glad", "pleased", "delighted", "thrilled", "fun",
            "joy", "exciting", "impressive", "outstanding", "superb"
        }
        self.negative_words = {
            "hate", "bad", "worst", "terrible", "horrible", "awful",
            "disgusting", "annoying", "stupid", "boring", "ugly",
            "pathetic", "miserable", "dreadful", "nightmare",
            "disaster", "failure", "useless", "worthless", "trash",
            "garbage", "crap", "sucks", "ridiculous", "absurd"
        }
        self.exaggeration_words = {
            "always", "never", "everyone", "nobody", "everything",
            "nothing", "everywhere", "forever", "all", "every",
            "entire", "completely", "absolute", "total", "only",
            "literally", "million", "billion", "infinity"
        }
        self.contrast_words = {
            "but", "however", "although", "though", "yet",
            "nevertheless", "despite", "whereas", "instead",
            "rather", "except", "still", "otherwise"
        }
        self.discourse_markers = {
            "because", "since", "so", "therefore", "thus",
            "meanwhile", "furthermore", "moreover", "anyway",
            "actually", "basically", "apparently", "supposedly",
            "honestly", "frankly", "clearly", "evidently"
        }
        self.rhetorical_starters = {
            "who", "what", "when", "where", "why", "how",
            "isn't", "aren't", "doesn't", "don't", "can't",
            "won't", "wouldn't", "shouldn't"
        }

    def extract(self, text):
        text = str(text)
        text_lower = text.lower()
        words = text_lower.split()
        wc = max(len(words), 1)
        features = []

        features.append(min(text.count("!"), 10))
        features.append(min(text.count("?"), 10))
        features.append(min(len(re.findall(r'\.{2,}', text)), 5))

        alpha = [c for c in text if c.isalpha()]
        caps_ratio = sum(1 for c in alpha if c.isupper()) / max(len(alpha), 1)
        features.append(caps_ratio)
        caps_words = sum(1 for w in text.split() if w.isupper() and len(w) > 1 and w.isalpha())
        features.append(min(caps_words, 10))

        features.append(min(wc, 100))
        features.append(np.mean([len(w) for w in words]) if words else 0)
        features.append(min(len(text), 500))
        features.append(sum(1 for c in text if c in '!?.,;:\'"-()') / max(len(text), 1))

        sarc = sum(1 for w in words if w in self.sarcasm_markers)
        features.append(min(sarc, 10))
        features.append(sarc / wc)

        intens = sum(1 for w in words if w in self.intensifiers)
        features.append(min(intens, 10))
        neg_count = sum(1 for w in words if w in self.negations)
        features.append(min(neg_count, 10))

        pos = sum(1 for w in words if w in self.positive_words)
        neg = sum(1 for w in words if w in self.negative_words)
        features.append(min(pos, 10))
        features.append(min(neg, 10))
        features.append(min(pos, neg) / wc)
        features.append(pos / max(neg, 1))

        exag = sum(1 for w in words if w in self.exaggeration_words)
        features.append(min(exag, 10))
        features.append(min(sum(1 for w in words if w in self.contrast_words), 5))
        features.append(min(sum(1 for w in words if w in self.discourse_markers), 5))

        features.append(1.0 if re.search(r'[!?]{2,}', text) else 0.0)
        features.append(min(text.count('"') + text.count("'"), 10))
        fp = {"i", "me", "my", "mine", "myself", "we", "us", "our"}
        features.append(min(sum(1 for w in words if w in fp), 10))
        features.append(1.0 if words and words[0] in self.sarcasm_markers else 0.0)
        features.append((sarc + intens + exag) / wc)

        features.append(1.0 if (words and words[0] in self.rhetorical_starters and "?" in text) else 0.0)
        features.append(1.0 if (pos > 0 and (neg_count > 0 or neg > 0)) else 0.0)
        features.append(1.0 if ("!" in text and pos > 0) else 0.0)
        features.append(1.0 if (wc <= 5 and sarc > 0) else 0.0)

        function_words = {
            "the", "a", "an", "is", "are", "was", "were", "be", "been",
            "being", "have", "has", "had", "do", "does", "did", "will",
            "would", "could", "should", "may", "might", "shall", "can",
            "to", "of", "in", "for", "on", "with", "at", "by", "from"
        }
        fw_count = sum(1 for w in words if w in function_words)
        features.append(fw_count / wc)

        return np.array(features, dtype=np.float32)

    def extract_batch(self, texts):
        return np.array([self.extract(str(t)) for t in texts])

# ══════════════════════════════════════════════════════════════
#  DATASET
# ══════════════════════════════════════════════════════════════

class SarcasmDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, feat_ext, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.feat_ext = feat_ext
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        enc = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt"
        )

        features = self.feat_ext.extract(text)

        return {
            "input_ids": enc["input_ids"].flatten(),
            "attention_mask": enc["attention_mask"].flatten(),
            "features": torch.tensor(features, dtype=torch.float32),
            "label": torch.tensor(label, dtype=torch.long)
        }

# ══════════════════════════════════════════════════════════════
#  HYBRID MODEL
# ══════════════════════════════════════════════════════════════

class HybridSarcasmModel(nn.Module):
    def __init__(self, model_name, num_features=30, dropout=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size

        for param in self.encoder.embeddings.parameters():
            param.requires_grad = False
        for i, layer in enumerate(self.encoder.encoder.layer):
            if i < 6:
                for param in layer.parameters():
                    param.requires_grad = False

        self.feat_net = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.GELU()
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden + 64, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 2)
        )

    def forward(self, input_ids, attention_mask, features):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        feat = self.feat_net(features)
        combined = torch.cat([cls, feat], dim=1)
        logits = self.classifier(combined)
        return logits

# ══════════════════════════════════════════════════════════════
#  STEP 1: LOAD DATASET
# ══════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("  STEP 1: LOADING DATASET")
print("=" * 60)

try:
    df = pd.read_csv(
        "final_dataset.csv",
        usecols=["text", "label"],
        engine="python",
        on_bad_lines="skip"
    )
except TypeError:
    df = pd.read_csv(
        "final_dataset.csv",
        usecols=["text", "label"],
        engine="python",
        error_bad_lines=False
    )

print(f"  Raw rows: {len(df)}")

df = df.dropna(subset=["text", "label"])
df["text"] = df["text"].astype(str)
df["label"] = df["label"].astype(int)

df = df[df["text"].str.len() >= 10]
df = df[df["text"].str.len() <= 250]
df = df[df["text"].str.split().str.len() >= 3]
df = df[df["label"].isin([0, 1])]

print(f"  After cleaning: {len(df)}")

# Since already balanced, use all or min(MAX_SAMPLES, len(df))
if len(df) > MAX_SAMPLES:
    per_class = MAX_SAMPLES // 2
    sarc_df = df[df["label"] == 1].sample(
        n=min(per_class, len(df[df["label"] == 1])),
        random_state=42
    )
    not_sarc_df = df[df["label"] == 0].sample(
        n=min(per_class, len(df[df["label"] == 0])),
        random_state=42
    )
    df = pd.concat([sarc_df, not_sarc_df]).reset_index(drop=True)

print(f"  Using: {len(df)} rows")

print(f"\n  Samples:")
for _, row in df.head(5).iterrows():
    t = str(row["text"])[:70]
    l = "SARC" if row["label"] == 1 else "NOT "
    print(f"    [{l}] {t}...")

texts = df["text"].tolist()
labels = df["label"].tolist()

train_t, val_t, train_l, val_l = train_test_split(
    texts, labels,
    test_size=0.15,
    random_state=42,
    stratify=labels
)

print(f"\n  Train: {len(train_t)} ({sum(train_l)} sarcastic)")
print(f"  Val:   {len(val_t)} ({sum(val_l)} sarcastic)")

# ══════════════════════════════════════════════════════════════
#  STEP 2: TRAIN MODEL
# ══════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("  STEP 2: TRAINING RoBERTa + LINGUISTIC FEATURES")
print("=" * 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
feat_ext = LinguisticFeatureExtractor()

train_ds = SarcasmDataset(train_t, train_l, tokenizer, feat_ext, MAX_LENGTH)
val_ds = SarcasmDataset(val_t, val_l, tokenizer, feat_ext, MAX_LENGTH)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

model = HybridSarcasmModel(MODEL_NAME, num_features=30).to(device)

trainable_params = [p for p in model.parameters() if p.requires_grad]
total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in trainable_params)
print(f"\n  Total params:     {total_params:,}")
print(f"  Trainable params: {train_params:,} ({train_params/total_params:.1%})")

optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=0.01, eps=1e-8)

total_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

best_f1 = 0
best_acc = 0
patience = 0
max_patience = 2

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    preds_all = []
    labels_all = []
    optimizer.zero_grad()

    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]")

    for step, batch in enumerate(progress):
        input_ids = batch["input_ids"].to(device)
        attn_mask = batch["attention_mask"].to(device)
        features = batch["features"].to(device)
        lbls = batch["label"].to(device)

        logits = model(input_ids, attn_mask, features)
        loss = criterion(logits, lbls) / GRADIENT_ACCUM
        loss.backward()

        if (step + 1) % GRADIENT_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * GRADIENT_ACCUM
        preds_all.extend(torch.argmax(logits, dim=1).cpu().numpy())
        labels_all.extend(lbls.cpu().numpy())

    train_acc = accuracy_score(labels_all, preds_all)
    train_f1 = f1_score(labels_all, preds_all, average="weighted")
    avg_train_loss = total_loss / len(train_loader)

    model.eval()
    val_preds = []
    val_true = []
    val_loss_total = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]"):
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)
            features = batch["features"].to(device)
            lbls = batch["label"].to(device)

            logits = model(input_ids, attn_mask, features)
            loss = criterion(logits, lbls)
            val_loss_total += loss.item()

            val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            val_true.extend(lbls.cpu().numpy())

    val_acc = accuracy_score(val_true, val_preds)
    val_f1 = f1_score(val_true, val_preds, average="weighted")
    avg_val_loss = val_loss_total / len(val_loader)

    print(f"\nEpoch {epoch+1}: Train [Loss={avg_train_loss:.4f}, Acc={train_acc:.4f}] | "
          f"Val [Loss={avg_val_loss:.4f}, Acc={val_acc:.4f}, F1={val_f1:.4f}]")

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_acc = val_acc
        torch.save(model.state_dict(), f"{SAVE_PATH}/model.pt")
        tokenizer.save_pretrained(SAVE_PATH)
        print("✓ BEST!")
        patience = 0
    else:
        patience += 1
        if patience >= max_patience and epoch >= 2:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

print(f"\nRoBERTa Training Complete!")
print(f"Best Val Acc: {best_acc:.4f}")
print(f"Best Val F1:  {best_f1:.4f}")

model.load_state_dict(torch.load(f"{SAVE_PATH}/model.pt", map_location=device))
model.eval()

val_preds_final = []
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attn_mask = batch["attention_mask"].to(device)
        features = batch["features"].to(device)
        logits = model(input_ids, attn_mask, features)
        val_preds_final.extend(torch.argmax(logits, dim=1).cpu().numpy())

print("\nClassification Report (RoBERTa + Features):")
print(classification_report(val_l, val_preds_final, target_names=["Not Sarcastic", "Sarcastic"]))

# ══════════════════════════════════════════════════════════════
#  STEP 3: TRAIN GB
# ══════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("  STEP 3: TRAINING GRADIENTBOOSTING")
print("=" * 60)

X_train = feat_ext.extract_batch(train_t)
X_val = feat_ext.extract_batch(val_t)

print("Getting RoBERTa predictions for feature augmentation...")
roberta_train_probs = []
temp_ds = SarcasmDataset(train_t, train_l, tokenizer, feat_ext, MAX_LENGTH)
temp_loader = DataLoader(temp_ds, batch_size=64, shuffle=False, num_workers=2)

with torch.no_grad():
    for batch in tqdm(temp_loader, desc="RoBERTa train probs"):
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        feats = batch["features"].to(device)
        logits = model(ids, mask, feats)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        roberta_train_probs.extend(probs)
roberta_train_probs = np.array(roberta_train_probs)

roberta_val_probs = []
with torch.no_grad():
    for batch in tqdm(val_loader, desc="RoBERTa val probs"):
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        feats = batch["features"].to(device)
        logits = model(ids, mask, feats)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        roberta_val_probs.extend(probs)
roberta_val_probs = np.array(roberta_val_probs)

X_train_aug = np.hstack([X_train, roberta_train_probs])
X_val_aug = np.hstack([X_val, roberta_val_probs])

gb = GradientBoostingClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    min_samples_leaf=10,
    random_state=42
)
gb.fit(X_train_aug, train_l)

gb_val_preds = gb.predict(X_val_aug)
gb_val_acc = accuracy_score(val_l, gb_val_preds)
gb_val_f1 = f1_score(val_l, gb_val_preds, average="weighted")

print(f"\nGB Val Acc: {gb_val_acc:.4f}")
print(f"GB Val F1:  {gb_val_f1:.4f}")

with open(f"{SAVE_PATH}/gb_model.pkl", "wb") as f:
    pickle.dump(gb, f)

# ══════════════════════════════════════════════════════════════
#  STEP 4: ENSEMBLE
# ══════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("  STEP 4: SMART ENSEMBLE CALIBRATION")
print("=" * 60)

deep_probs = roberta_val_probs
shallow_probs = gb.predict_proba(X_val_aug)

best_w = 0.7
best_ens_f1 = 0
best_ens_acc = 0

for w in np.arange(0.40, 0.95, 0.05):
    ens = w * deep_probs + (1 - w) * shallow_probs
    preds = np.argmax(ens, axis=1)
    f = f1_score(val_l, preds, average="weighted")
    a = accuracy_score(val_l, preds)
    marker = " ◀ BEST" if f > best_ens_f1 else ""
    print(f"RoBERTa={w:.0%} + GB={1-w:.0%}: Acc={a:.4f}, F1={f:.4f}{marker}")
    if f > best_ens_f1:
        best_ens_f1 = f
        best_ens_acc = a
        best_w = w

with open(f"{SAVE_PATH}/ensemble_config.pkl", "wb") as f:
    pickle.dump({"deep_weight": best_w, "shallow_weight": 1 - best_w}, f)

deep_acc = accuracy_score(val_l, np.argmax(deep_probs, axis=1))
deep_f1 = f1_score(val_l, np.argmax(deep_probs, axis=1), average="weighted")

print(f"\nRoBERTa+Features Acc={deep_acc:.4f}, F1={deep_f1:.4f}")
print(f"GB Acc={gb_val_acc:.4f}, F1={gb_val_f1:.4f}")
print(f"ENSEMBLE ({best_w:.0%}+{1-best_w:.0%}) Acc={best_ens_acc:.4f}, F1={best_ens_f1:.4f}")

ens_final = best_w * deep_probs + (1 - best_w) * shallow_probs
ens_preds = np.argmax(ens_final, axis=1)

print("\nEnsemble Report:")
print(classification_report(val_l, ens_preds, target_names=["Not Sarcastic", "Sarcastic"]))

# ══════════════════════════════════════════════════════════════
#  STEP 5: CLAUSE + SENTIMENT
# ══════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("  STEP 5: LOADING CLAUSE & SENTIMENT")
print("=" * 60)

nlp_spacy = spacy.load("en_core_web_sm")
nlp_stanza = stanza.Pipeline("en", processors="tokenize,pos,constituency", verbose=False)

sent_tok = AutoTokenizer.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment-latest")
sent_mdl = AutoModelForSequenceClassification.from_pretrained(
    "cardiffnlp/twitter-roberta-base-sentiment-latest"
).to(device)
sent_mdl.eval()
sent_labels = {0: "negative", 1: "neutral", 2: "positive"}

def predict_sarcasm(text):
    text = str(text)
    enc = tokenizer(
        text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    ids = enc["input_ids"].to(device)
    mask = enc["attention_mask"].to(device)
    feats_tensor = torch.tensor(feat_ext.extract(text), dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(ids, mask, feats_tensor)
        dp = torch.softmax(logits, dim=1).cpu().numpy()[0]

    ling_feats = feat_ext.extract(text).reshape(1, -1)
    aug_feats = np.hstack([ling_feats, dp.reshape(1, -1)])
    sp = gb.predict_proba(aug_feats)[0]

    ep = best_w * dp + (1 - best_w) * sp
    pred = int(np.argmax(ep))

    return {
        "is_sarcastic": pred == 1,
        "confidence": float(ep[pred]),
        "probs": {"not_sarcastic": float(ep[0]), "sarcastic": float(ep[1])},
        "deep": {"not_sarcastic": float(dp[0]), "sarcastic": float(dp[1])},
        "shallow": {"not_sarcastic": float(sp[0]), "sarcastic": float(sp[1])}
    }

def classify_clause(sentence):
    sentence = str(sentence)
    doc = nlp_spacy(sentence)

    coord = {"for", "and", "nor", "but", "or", "yet", "so"}
    sub_m = {"because", "since", "although", "if", "when", "while",
             "though", "unless", "until", "before", "after", "that",
             "whether", "whereas", "even", "once", "so"}

    cc = sum(1 for t in doc if t.dep_ == "cc" and t.text.lower() in coord and t.head.dep_ in ("ROOT", "conj"))
    dep_types = {"advcl", "relcl", "ccomp", "acl", "csubj", "csubjpass"}
    dep = sum(1 for t in doc if t.dep_ in dep_types)

    for t in doc:
        if t.dep_ == "mark" and t.text.lower() in sub_m:
            dep = max(dep, 1)

    ind = max(cc + 1, 1)

    try:
        sd = nlp_stanza(sentence)
        for s in sd.sentences:
            ts = str(s.constituency)
            sbar = ts.count("(SBAR")
            if sbar > dep:
                dep = sbar
            sc = ts.count("(S ") + ts.count("(S\n")
            if sc > 1 and sc > ind:
                ind = sc
    except:
        pass

    ind_cl, dep_cl = [], []
    root = None
    for t in doc:
        if t.dep_ == "ROOT":
            root = t
            break

    if root:
        sub = sorted([t.i for t in root.subtree])
        if sub:
            ind_cl.append(doc[sub[0]:sub[-1]+1].text)
        for t in doc:
            if t.dep_ == "conj" and t.head == root:
                sub = sorted([t.i for t in t.subtree])
                if sub:
                    ind_cl.append(doc[sub[0]:sub[-1]+1].text)

    if not ind_cl:
        ind_cl = [sentence]

    for t in doc:
        if t.dep_ in dep_types and t.pos_ in ("VERB", "AUX"):
            sub = sorted([t.i for t in t.subtree])
            if sub:
                dep_cl.append(doc[sub[0]:sub[-1]+1].text)

    if ind >= 2 and dep >= 1:
        ct = "compound-complex"
    elif ind >= 2:
        ct = "compound"
    elif dep >= 1:
        ct = "complex"
    else:
        ct = "simple"

    return {"type": ct, "n_ind": ind, "n_dep": dep, "ind_cl": ind_cl, "dep_cl": dep_cl}

def predict_sentiment(text):
    enc = sent_tok(
        str(text),
        max_length=128,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    ids = enc["input_ids"].to(device)
    mask = enc["attention_mask"].to(device)
    with torch.no_grad():
        logits = sent_mdl(input_ids=ids, attention_mask=mask).logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred = int(np.argmax(probs))
    return sent_labels[pred], float(probs[pred])

def full_analysis(text):
    sarc = predict_sarcasm(text)

    # Clause classification only for sarcastic texts
    if not sarc["is_sarcastic"]:
        return {
            "text": text,
            "sarcasm": sarc,
            "clause": None,
            "sentiments": [],
            "overall": "N/A"
        }

    clause = classify_clause(text)

    results = []
    for c in clause["ind_cl"]:
        if c.strip():
            s, sc = predict_sentiment(c)
            results.append({"text": c, "role": "independent", "sentiment": s, "conf": sc})

    for c in clause["dep_cl"]:
        if c.strip():
            s, sc = predict_sentiment(c)
            results.append({"text": c, "role": "dependent", "sentiment": s, "conf": sc})

    if not results:
        s, sc = predict_sentiment(text)
        results.append({"text": text, "role": "independent", "sentiment": s, "conf": sc})

    overall = Counter([r["sentiment"] for r in results]).most_common(1)[0][0]

    return {"text": text, "sarcasm": sarc, "clause": clause, "sentiments": results, "overall": overall}

def print_result(r):
    print("\n" + "=" * 70)
    print(f"Input: \"{r['text']}\"")
    print("-" * 70)

    s = r["sarcasm"]
    icon = "SARCASTIC ✓" if s["is_sarcastic"] else "NOT SARCASTIC ✗"
    print(f"SARCASM: {icon} ({s['confidence']:.1%})")
    print(f"  RoBERTa: {s['deep']['sarcastic']:.1%}")
    print(f"  GB:      {s['shallow']['sarcastic']:.1%}")
    print(f"  Ensemble:{s['probs']['sarcastic']:.1%}")

    if not s["is_sarcastic"]:
        print("\nCLAUSE: Skipped (only for sarcastic texts)")
        print("=" * 70)
        return

    c = r["clause"]
    print(f"\nCLAUSE: {c['type'].upper()} ({c['n_ind']} ind, {c['n_dep']} dep)")

    print("\nSENTIMENT:")
    for i, cl in enumerate(r["sentiments"], 1):
        sym = {"positive": "[+]", "negative": "[-]", "neutral": "[~]"}.get(cl["sentiment"], "[?]")
        print(f"  {i}. [{cl['role']}] \"{cl['text']}\"")
        print(f"     {sym} {cl['sentiment'].upper()} ({cl['conf']:.1%})")

    print(f"\nOVERALL: {r['overall'].upper()}")
    print("=" * 70)

# ══════════════════════════════════════════════════════════════
#  STEP 6: TEST PIPELINE
# ══════════════════════════════════════════════════════════════

print("\n\n" + "#" * 70)
print("HYBRID PIPELINE TEST RESULTS")
print("#" * 70)

tests = [
    "Oh great, another Monday morning. Just what I needed.",
    "Yeah right, because that's totally how science works.",
    "Sure, because working overtime without pay is everyone's dream.",
    "The team delivered the project on time and the client was satisfied.",
    "I enjoyed the movie because the storyline was compelling.",
    "I love deadlines and I especially enjoy missing them.",
    "What a surprise, the train is late again even though they promised improvements, and now I'll miss my meeting because nobody cares.",
    "Great idea!",
    "Yeah because that's definitely going to work.",
    "The weather is lovely today and I enjoyed my walk.",
    "Nothing like a cold cup of coffee to start the day perfectly.",
    "I could use one of those tools."
]

for text in tests:
    r = full_analysis(text)
    print_result(r)

print("\n\n" + "=" * 115)
print(f"{'Text':<50} | {'Sarcasm?':<12} | {'Conf':<6} | {'Clause':<18} | {'Sentiment':<10}")
print("-" * 115)

for text in tests:
    r = full_analysis(text)
    short = text9[:47] + "..." if len(text) > 47 else text
    s = "YES ✓" if r["sarcasm"]["is_sarcastic"] else "NO ✗"
    clause_type = r["clause"]["type"] if r["clause"] else "Skipped"
    overall = r["overall"]
    print(f"{short:<50} | {s:<12} | {r['sarcasm']['confidence']:.0%}    | {clause_type:<18} | {overall:<10}")

print("=" * 115)

print(f"\n\n{'=' * 60}")
print("PIPELINE COMPLETE!")
print("=" * 60)
print(f"Models: RoBERTa + 30 Features + GradientBoosting")
print(f"Ensemble: {best_w:.0%} RoBERTa + {1-best_w:.0%} GB")
print(f"RoBERTa Acc:    {deep_acc:.2%}")
print(f"GB Acc:         {gb_val_acc:.2%}")
print(f"Ensemble Acc:   {best_ens_acc:.2%}")
print(f"Saved to: {SAVE_PATH}/")
print("=" * 60)

Device: cuda
GPU: Tesla T4

Settings:
  Model:      roberta-base
  Samples:    29970
  Epochs:     5
  Batch:      32 (effective: 64)
  Max Length: 128
  LR:         2e-05

  STEP 1: LOADING DATASET
  Raw rows: 29970
  After cleaning: 27914
  Using: 27914 rows

  Samples:
    [NOT ] texas high school coach allegedly told black students 'i'm going to ha...
    [NOT ] james blake says cop who slammed him 'doesn't deserve' to have badge...
    [NOT ] gop race heads to south carolina, known for dirty tricks and brawls...
    [NOT ] scott pruitt lands a second fawning conservative magazine profile...
    [NOT ] rand paul ends daylong nsa 'filibuster'...

  Train: 23726 (11051 sarcastic)
  Val:   4188 (1951 sarcastic)

  STEP 2: TRAINING RoBERTa + LINGUISTIC FEATURES


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.




######################################################################
HYBRID PIPELINE TEST RESULTS
######################################################################

Input: "Oh great, another Monday morning. Just what I needed."
----------------------------------------------------------------------
SARCASM: SARCASTIC ✓ (95.1%)
  RoBERTa: 94.5%
  GB:      100.0%
  Ensemble:95.1%

CLAUSE: SIMPLE (1 ind, 0 dep)

SENTIMENT:
  1. [independent] "Oh great, another Monday morning."
     [+] POSITIVE (61.4%)

OVERALL: POSITIVE

Input: "Yeah right, because that's totally how science works."
----------------------------------------------------------------------
SARCASM: SARCASTIC ✓ (95.9%)
  RoBERTa: 95.4%
  GB:      100.0%
  Ensemble:95.9%

CLAUSE: COMPOUND-COMPLEX (3 ind, 2 dep)

SENTIMENT:
  1. [independent] "Yeah right, because that's totally how science works."
     [+] POSITIVE (64.4%)
  2. [dependent] "because that's totally how science works"
     [+] POSITIVE (55.1%)
  3. [depend

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]


Input: "I enjoyed the movie because the storyline was compelling."
----------------------------------------------------------------------
SARCASM: SARCASTIC ✓ (95.5%)
  RoBERTa: 95.0%
  GB:      100.0%
  Ensemble:95.5%

CLAUSE: COMPOUND-COMPLEX (2 ind, 1 dep)

SENTIMENT:
  1. [independent] "I enjoyed the movie because the storyline was compelling."
     [+] POSITIVE (98.4%)
  2. [dependent] "because the storyline was compelling"
     [+] POSITIVE (85.0%)

OVERALL: POSITIVE

Input: "I love deadlines and I especially enjoy missing them."
----------------------------------------------------------------------
SARCASM: SARCASTIC ✓ (84.8%)
  RoBERTa: 83.1%
  GB:      99.9%
  Ensemble:84.8%

CLAUSE: COMPOUND (4 ind, 0 dep)

SENTIMENT:
  1. [independent] "I love deadlines and I especially enjoy missing them."
     [+] POSITIVE (97.3%)
  2. [independent] "I especially enjoy missing them."
     [+] POSITIVE (94.5%)

OVERALL: POSITIVE

Input: "What a surprise, the train is late again even though

In [ ]:
import os

print(os.path.abspath("./saved_models/hybrid_v2"))

/content/saved_models/hybrid_v2


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import shutil

src = "/content/saved_models/hybrid_v2"
dst = "/content/drive/MyDrive/saved_models/hybrid_v2"

os.makedirs("/content/drive/MyDrive/saved_models", exist_ok=True)

if os.path.exists(dst):
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("✅ Model copied to:", dst)
print("Files:", os.listdir(dst))

✅ Model copied to: /content/drive/MyDrive/saved_models/hybrid_v2
Files: ['ensemble_config.pkl', 'model.pt', 'tokenizer_config.json', 'tokenizer.json', 'gb_model.pkl']
